# 01 · Training Library

Defines the model, training procedure, and evaluation routines shared by all
training notebooks. The architecture is a ConvNeXt-Large backbone with stage
freezing and differential learning rates, a hierarchical classification head, and
an auxiliary domain classifier connected through a gradient reversal layer for
domain-adversarial training. Image data is loaded once per session from a single
packed array and held in memory; pixels are read by index, eliminating per-image
file access and decoding. Includes joint-region cropping, phase-aware sampling,
noise-aware and label-quality weighting, curriculum learning, mixed-precision
optimization with gradient accumulation, per-epoch checkpointing, test-time
augmentation, and per-image prediction export.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
print('Config loaded. Model:', config.MODEL_NAME)

Mounted at /content/drive
Config loaded. Model: convnext_large


## Write training library

In [ ]:
import sys
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
lib_code = r'''# Scope 3 Training Library — ConvNeXt-Large, array-based loading
import os, json, time, math, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.autograd import Function
from PIL import Image
import config

DATASET_TO_IDX = {"oai":0,"nhanes3":1,"mrkr":2,"mendeley":3}
_IMAGES = None  # global in-memory array (N,224,224) uint8

def joint_crop(arr):
    if not getattr(config, "USE_JOINT_CROP", False): return arr
    h, w = arr.shape[:2]; f = getattr(config, "JOINT_CROP_FRAC", 0.80)
    ch, cw = int(h*f), int(w*f); top, left = (h-ch)//2, (w-cw)//2
    return arr[top:top+ch, left:left+cw]

def prepare_local_data(force=False):
    global _IMAGES
    import shutil, threading
    import concurrent.futures as cf
    man = pd.read_csv(str(config.MANIFEST_CSV))
    local_npy = "/content/images.npy"
    if _IMAGES is not None and not force and len(_IMAGES) >= len(man)*0.99:
        print(f"Image array already loaded ({len(_IMAGES):,}) - reusing.")
        if "arr_idx" not in man.columns: man["arr_idx"]=np.arange(len(man))
        return man
    if os.path.exists(str(config.IMAGES_NPY)):
        try:
            if not os.path.exists(local_npy):
                t0=time.time(); shutil.copy(str(config.IMAGES_NPY), local_npy); print(f"Copied array in {time.time()-t0:.0f}s")
            t1=time.time(); _IMAGES=np.load(local_npy); print(f"Loaded array {tuple(_IMAGES.shape)} in {time.time()-t1:.0f}s")
            if "arr_idx" not in man.columns: man["arr_idx"]=np.arange(len(man))
            return man
        except Exception as e: print(f"Array path failed ({e}) - building from PNGs.")
    S=config.IMG_SIZE; N=len(man); rows=man.reset_index(drop=True)
    paths=[str(config.DATASETS[r["dataset"]]["png_dir"]/r["filename"]) for _,r in rows.iterrows()]
    arr=np.empty((N,S,S),dtype=np.uint8); done=[0]; lock=threading.Lock()
    print(f"Building array from {N:,} PNGs (16 workers)..."); t0=time.time()
    def load(i):
        try:
            im=Image.open(paths[i]).convert("L")
            if im.size!=(S,S): im=im.resize((S,S),Image.LANCZOS)
            arr[i]=np.array(im,dtype=np.uint8)
            with lock:
                done[0]+=1
                if done[0]%10000==0: print(f"  {done[0]:,}/{N:,}",flush=True)
            return True
        except Exception: return False
    with cf.ThreadPoolExecutor(max_workers=16) as ex: list(ex.map(load,range(N)))
    print(f"Built in {time.time()-t0:.0f}s"); _IMAGES=arr
    if "arr_idx" not in man.columns: man["arr_idx"]=np.arange(len(man))
    return man

def _get_image(row):
    global _IMAGES
    if _IMAGES is not None and "arr_idx" in row: return _IMAGES[int(row["arr_idx"])]
    im=Image.open(row["abs_path"]).convert("L")
    if im.size!=(config.IMG_SIZE,config.IMG_SIZE): im=im.resize((config.IMG_SIZE,config.IMG_SIZE),Image.LANCZOS)
    return np.array(im,dtype=np.uint8)

def load_quality_weights():
    q={}
    try:
        df=pd.read_csv(str(config.LABEL_QUALITY_CSV)); lo=config.QUALITY_MIN_WEIGHT
        for _,r in df.iterrows(): q[r["filename"]]=max(lo,min(1.0, lo+(1.0-lo)*float(r["label_quality"])))
    except Exception: pass
    return q

class KneeDataset(Dataset):
    def __init__(self, df, train=True, quality=None):
        self.df=df.reset_index(drop=True); self.train=train; self.quality=quality or {}
    def __len__(self): return len(self.df)
    def _aug(self,a):
        if not self.train: return a
        import cv2
        if random.random()<0.5: a=np.fliplr(a).copy()
        if random.random()<0.5:
            ang=random.uniform(-7,7); h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),ang,1.0)
            a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
        if random.random()<0.3:
            f=random.uniform(0.9,1.1); a=np.clip(a.astype(np.float32)*f,0,255).astype(np.uint8)
        return a
    def __getitem__(self,i):
        r=self.df.iloc[i]; a=joint_crop(_get_image(r))
        if a.shape!=(config.IMG_SIZE,config.IMG_SIZE):
            import cv2; a=cv2.resize(a,(config.IMG_SIZE,config.IMG_SIZE),interpolation=cv2.INTER_AREA)
        a=self._aug(a)
        x=torch.from_numpy(a.astype(np.float32)/255.0); x=(x-0.485)/0.229; x=x.unsqueeze(0).repeat(3,1,1)
        y=int(r["kl_grade"]); base=config.LOSS_WEIGHTS.get(r["kl_source"],1.0)
        qw=self.quality.get(r["filename"],1.0) if r["kl_source"]=="model_predicted" else 1.0
        return x, y, torch.tensor(base*qw,dtype=torch.float32), DATASET_TO_IDX.get(r["dataset"],0)

def build_sampler(df, mrkr_weight=1.0, clean_only=False):
    counts=df["dataset"].value_counts().to_dict(); w=np.zeros(len(df))
    src=df["kl_source"].values; ds=df["dataset"].values
    for i in range(len(df)):
        b=1.0/counts[ds[i]]
        if src[i]=="model_predicted": b=0.0 if clean_only else b*mrkr_weight
        w[i]=b
    if w.sum()==0: w[:]=1.0
    return WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),len(df),replacement=True)

class GradReverse(Function):
    @staticmethod
    def forward(ctx,x,l): ctx.l=l; return x.view_as(x)
    @staticmethod
    def backward(ctx,g): return g.neg()*ctx.l, None
def grad_reverse(x,l=1.0): return GradReverse.apply(x,l)

class HierarchicalNet(nn.Module):
    def __init__(self, num_classes=5, n_domains=4, use_hierarchical=True):
        super().__init__(); self.use_hierarchical=use_hierarchical
        from torchvision import models
        try:
            from torchvision.models import ConvNeXt_Large_Weights
            self.backbone=models.convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)
        except Exception: self.backbone=models.convnext_large(weights="IMAGENET1K_V1")
        feat=self.backbone.classifier[2].in_features
        self.backbone.classifier=nn.Identity(); self.feat_dim=feat
        self.head_5=nn.Sequential(nn.Flatten(1),nn.LayerNorm(feat),nn.Dropout(0.3),nn.Linear(feat,num_classes))
        self.head_s1=nn.Sequential(nn.Flatten(1),nn.Dropout(0.3),nn.Linear(feat,2))
        self.head_s2=nn.Sequential(nn.Flatten(1),nn.Dropout(0.3),nn.Linear(feat,2))
        self.domain_head=nn.Sequential(nn.Flatten(1),nn.Dropout(0.3),nn.Linear(feat,128),nn.ReLU(),nn.Linear(128,n_domains))
    def forward(self,x,grl_lambda=0.0):
        f=self.backbone(x)
        if f.dim()==4: f=f.mean(dim=[-2,-1])
        return self.head_5(f), self.head_s1(f), self.head_s2(f), self.domain_head(grad_reverse(f,grl_lambda))
    def freeze_stages(self, n_freeze):
        for idx,block in enumerate(self.backbone.features):
            req=idx>=(n_freeze*2)
            for p in block.parameters(): p.requires_grad=req
    def param_groups(self):
        hp,bp=[],[]
        for n,p in self.named_parameters():
            if p.requires_grad: (hp if "head" in n else bp).append(p)
        return hp,bp

def hier_loss(o5,s1,s2,y,lw,use_hierarchical=True,use_ordinal=False):
    if use_ordinal:
        # Distance-weighted ordinal cross-entropy: standard CE scaled by how far the
        # predicted-distribution mass sits from the true rank. Trains the 5-way head,
        # decodes with argmax (matches predict_tta), and respects ordinality.
        K=o5.shape[1]
        logp=F.log_softmax(o5,dim=1)
        ce_base=F.nll_loss(logp,y,reduction="none")
        ranks=torch.arange(K,device=y.device).float().unsqueeze(0)
        probs=logp.exp()
        exp_rank=(probs*ranks).sum(1)
        dist=(exp_rank-y.float()).abs()
        ce=ce_base*(1.0+dist)
    else: ce=F.cross_entropy(o5,y,reduction="none")
    tot=ce.clone()
    if use_hierarchical:
        t1=(y>=config.STAGE1_OA_THRESHOLD).long(); c1=F.cross_entropy(s1,t1,reduction="none")
        oa=t1.bool(); c2=torch.zeros_like(ce)
        if oa.any(): c2[oa]=F.cross_entropy(s2[oa],(y[oa]==4).long(),reduction="none")
        tot=tot+0.5*c1+0.5*c2
    return (tot*lw).mean()
def domain_loss(d,t): return F.cross_entropy(d,t)

def save_ckpt(p,m,o,e,bv,h):
    t=str(p)+".tmp"; torch.save({"model":m.state_dict(),"opt":o.state_dict(),"epoch":e,"best_val":bv,"history":h},t); os.replace(t,str(p))
def load_ckpt(p,m,o):
    ck=torch.load(str(p),map_location="cpu"); m.load_state_dict(ck["model"])
    if o is not None and "opt" in ck: o.load_state_dict(ck["opt"])
    return ck["epoch"],ck.get("best_val",0.0),ck.get("history",[])

@torch.no_grad()
def predict_tta(model,df,device,batch_size=32,use_tta=True):
    import cv2; model.eval(); df=df.reset_index(drop=True); preds=[]; probs=[]
    def ln(a):
        x=torch.from_numpy(a.astype(np.float32)/255.0); x=(x-0.485)/0.229; return x.unsqueeze(0).repeat(3,1,1)
    tfs=config.TTA_TRANSFORMS if use_tta else ["identity"]
    for s in range(0,len(df),batch_size):
        sub=df.iloc[s:s+batch_size]; acc=None
        for tn in tfs:
            b=[]
            for _,r in sub.iterrows():
                a=joint_crop(_get_image(r))
                if a.shape!=(config.IMG_SIZE,config.IMG_SIZE): a=cv2.resize(a,(config.IMG_SIZE,config.IMG_SIZE),interpolation=cv2.INTER_AREA)
                if tn=="hflip": a=np.fliplr(a).copy()
                elif tn=="rot+5":
                    h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),5,1.0); a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
                elif tn=="rot-5":
                    h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),-5,1.0); a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
                b.append(ln(a))
            xb=torch.stack(b).to(device); o5,_,_,_=model(xb,grl_lambda=0.0)
            p=F.softmax(o5,1).cpu().numpy(); acc=p if acc is None else acc+p
        acc/=len(tfs); probs.append(acc); preds.extend(acc.argmax(1).tolist())
    return np.array(preds), np.vstack(probs)

def compute_metrics(yt,yp,probs=None):
    from sklearn.metrics import accuracy_score,cohen_kappa_score,roc_auc_score,f1_score
    yt=np.asarray(yt); yp=np.asarray(yp); m={}
    m["acc5"]=float(accuracy_score(yt,yp)); m["qwk"]=float(cohen_kappa_score(yt,yp,weights="quadratic"))
    bt=(yt>=2).astype(int); bp=(yp>=2).astype(int); m["acc_binary"]=float(accuracy_score(bt,bp))
    mk=np.isin(yt,[1,2]); m["f1_kl12"]=float(f1_score(yt[mk],np.clip(yp[mk],1,2),labels=[1,2],average="macro")) if mk.sum()>0 else float("nan")
    if probs is not None:
        try: m["auc_oa"]=float(roc_auc_score(bt,probs[:,2:].sum(1)))
        except Exception: m["auc_oa"]=float("nan")
        oa=yt>=2
        if oa.sum()>10 and len(np.unique((yt[oa]==4)))>1:
            try: m["auc_severity"]=float(roc_auc_score((yt[oa]==4).astype(int),probs[oa,4]/(probs[oa,2:].sum(1)+1e-8)))
            except Exception: m["auc_severity"]=float("nan")
        else: m["auc_severity"]=float("nan")
    return m

def run_training(run_name,train_df,val_df,test_df,mechanisms,seed,ckpt_dir,results_dir,log_fn=print):
    config.set_seed(seed)
    if not torch.cuda.is_available(): raise RuntimeError("No GPU. Change runtime type to GPU.")
    device="cuda"; done=os.path.join(str(results_dir),run_name+".json")
    if os.path.exists(done):
        with open(done) as f: log_fn(f"  [{run_name}] complete - skip."); return json.load(f)
    use_hier=mechanisms.get("hierarchical",True); use_ord=mechanisms.get("ordinal",False)
    use_dom=mechanisms.get("domain_adv",False); use_q=mechanisms.get("use_quality",False)
    quality=load_quality_weights() if use_q else {}
    model=HierarchicalNet(config.NUM_CLASSES,4,use_hier).to(device)
    model.freeze_stages(config.FREEZE_STAGES); hp,bp=model.param_groups()
    opt=torch.optim.AdamW([{"params":hp,"lr":config.LR_HEAD},{"params":bp,"lr":config.LR_BACKBONE}],weight_decay=config.WEIGHT_DECAY)
    scaler=torch.amp.GradScaler("cuda"); ckpt=os.path.join(str(ckpt_dir),run_name+".pt")
    e0,best,hist=0,0.0,[]
    if os.path.exists(ckpt): e0,best,hist=load_ckpt(ckpt,model,opt); log_fn(f"  [{run_name}] resume ep{e0}")
    no_imp=0; MIN_DELTA=0.005; accum=config.GRAD_ACCUM
    for epoch in range(e0,config.EPOCHS):
        if mechanisms.get("curriculum",False):
            if epoch<config.CURRICULUM["phase1_end"]: clean,mw=True,0.0
            elif epoch<config.CURRICULUM["phase2_end"]:
                clean=False; fr=(epoch-config.CURRICULUM["phase1_end"])/max(1,config.CURRICULUM["phase2_end"]-config.CURRICULUM["phase1_end"]); mw=fr*config.CURRICULUM["mrkr_target_weight"]
            else: clean,mw=False,config.CURRICULUM["mrkr_target_weight"]
        else: clean,mw=False,config.CURRICULUM["mrkr_target_weight"]
        if mechanisms.get("sampler",False):
            loader=DataLoader(KneeDataset(train_df,True,quality),batch_size=config.BATCH_SIZE,sampler=build_sampler(train_df,mw,clean),num_workers=config.NUM_WORKERS,pin_memory=True)
        else:
            sub=train_df[train_df["kl_source"]!="model_predicted"] if clean else train_df
            loader=DataLoader(KneeDataset(sub,True,quality),batch_size=config.BATCH_SIZE,shuffle=True,num_workers=config.NUM_WORKERS,pin_memory=True)
        lr_scale=(epoch+1)/config.WARMUP_EPOCHS if epoch<config.WARMUP_EPOCHS else 0.5*(1+math.cos(math.pi*(epoch-config.WARMUP_EPOCHS)/max(1,config.EPOCHS-config.WARMUP_EPOCHS)))
        opt.param_groups[0]["lr"]=config.LR_HEAD*lr_scale; opt.param_groups[1]["lr"]=config.LR_BACKBONE*lr_scale
        _lmax=mechanisms.get("grl_lambda_max", getattr(config,"GRL_LAMBDA_MAX",1.0))
        grl=(_lmax*(2.0/(1.0+math.exp(-10*epoch/max(1,config.EPOCHS)))-1.0)) if use_dom else 0.0
        model.train(); t0=time.time(); rl=0.0; nb=0; tc=0; tt=0; use_noise=mechanisms.get("noise_loss",False); opt.zero_grad()
        for bi,(x,y,lw,dom) in enumerate(loader):
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
            lw=lw.to(device) if (use_noise or use_q) else torch.ones_like(lw).to(device); dom=dom.to(device)
            with torch.amp.autocast("cuda"):
                o5,s1,s2,dl=model(x,grl_lambda=grl); loss=hier_loss(o5,s1,s2,y,lw,use_hier,use_ord)
                if use_dom: loss=loss+domain_loss(dl,dom)
                loss=loss/accum
            scaler.scale(loss).backward()
            if (bi+1)%accum==0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad()
            rl+=loss.item()*accum; nb+=1
            with torch.no_grad(): tc+=(o5.argmax(1)==y).sum().item(); tt+=y.size(0)
        tra=tc/max(1,tt)
        vp,vpr=predict_tta(model,val_df,device,config.BATCH_SIZE,use_tta=False); vm=compute_metrics(val_df["kl_grade"].values,vp,vpr); gap=tra-vm["acc5"]
        hist.append({"epoch":epoch,"loss":rl/max(1,nb),"train_acc":tra,"train_val_gap":gap,"grl":grl,**vm})
        log_fn(f"  [{run_name}] ep{epoch+1}/{config.EPOCHS} loss={rl/max(1,nb):.3f} tr={tra:.3f} val={vm['acc5']:.3f} gap={gap:+.3f} qwk={vm['qwk']:.3f} grl={grl:.2f} ({time.time()-t0:.0f}s){'  <-- OVERFIT?' if gap>0.15 else ''}")
        imp=vm["acc5"]>(best+MIN_DELTA)
        if imp or vm["acc5"]>best:
            if vm["acc5"]>best: best=vm["acc5"]; save_ckpt(os.path.join(str(ckpt_dir),run_name+"_best.pt"),model,opt,epoch,best,hist)
            no_imp=0 if imp else no_imp+1
        else: no_imp+=1
        save_ckpt(ckpt,model,opt,epoch+1,best,hist)
        if no_imp>=config.EARLY_STOP_PATIENCE: log_fn(f"  [{run_name}] early stop ep{epoch+1}"); break
    bpath=os.path.join(str(ckpt_dir),run_name+"_best.pt")
    if os.path.exists(bpath): load_ckpt(bpath,model,None)
    tp,tpr=predict_tta(model,test_df,device,config.BATCH_SIZE,use_tta=config.USE_TTA); tm=compute_metrics(test_df["kl_grade"].values,tp,tpr); vb=max((h["acc5"] for h in hist),default=0.0)
    if getattr(config,"SAVE_PREDICTIONS",False):
        np.savez_compressed(os.path.join(str(results_dir),run_name+"_preds.npz"),y_true=test_df["kl_grade"].values,y_pred=tp,probs=tpr)
    res={"run_name":run_name,"seed":seed,"mechanisms":mechanisms,"internal_acc5":vb,"external_acc5":tm["acc5"],
         "external_qwk":tm["qwk"],"external_acc_binary":tm["acc_binary"],"external_auc_oa":tm.get("auc_oa",float("nan")),
         "external_auc_severity":tm.get("auc_severity",float("nan")),"external_f1_kl12":tm.get("f1_kl12",float("nan")),
         "gap":vb-tm["acc5"],"final_train_acc":hist[-1]["train_acc"] if hist else float("nan"),
         "max_train_val_gap":max((h.get("train_val_gap",0) for h in hist),default=0.0),
         "overfit_flag":bool(max((h.get("train_val_gap",0) for h in hist),default=0.0)>0.15),
         "n_train":len(train_df),"n_val":len(val_df),"n_test":len(test_df),"history":hist}
    with open(done,"w") as f: json.dump(res,f,indent=2)
    log_fn(f"  [{run_name}] DONE int={vb:.3f} ext={tm['acc5']:.3f} gap={res['gap']:.3f} qwk={tm['qwk']:.3f}")
    return res
'''
p='/content/drive/MyDrive/Master Thesis/scope3/training_lib.py'
with open(p,'w') as f: f.write(lib_code)
print(f'training_lib.py written: {len(lib_code):,} bytes')
import importlib
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
print('Imports OK. Loading: single array in memory, pixels read by index.')

training_lib.py written: 16,771 bytes
Imports OK. Loading: single array in memory, pixels read by index.
